# Experiment 10 — TFT market forecast + listing-level residual

This turns Experiment 9's segment-month TFT forecasts into a prediction for every listing. The model is additive in log space: `log(rent) = log(TFT market rent) + listing residual`. The residual model uses the leakage-aware `compact` listing features.

Validation is chronological. The TFT validation forecast is trained only through 2024-03 and predicts 2024-04 onward; the residual model is also fit only through 2024-03. Blend weights are selected exclusively on this holdout.

**Executed result:** The direct compact listing model scored RMSE **471.209**, MAE **349.000**, and R² **0.4890**. TFT plus the listing residual scored RMSE **499.791**, and the optimized blend assigned the TFT path weight **0.00**, reproducing the direct model's 471.209 RMSE. The strong segment-level TFT score therefore did not translate into a useful listing-level improvement.

In [1]:
from pathlib import Path
import os
import sys
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from autogluon.tabular import TabularPredictor

configured_root = os.environ.get('KDDM_PROJECT_ROOT')
roots = ([Path(configured_root)] if configured_root else []) + [Path.cwd(), *Path.cwd().parents]
project_root = next((p for p in roots if (p / 'data/train.csv').exists()), None)
if project_root is None:
    raise FileNotFoundError('Set KDDM_PROJECT_ROOT to the repository root.')
experiment_dir = project_root / 'exploration' / 'metul'
sys.path.insert(0, str(experiment_dir))
from preprocessing import RentalPreprocessor, categorical_clean, load_datasets

TARGET = 'MONTHLY_RENT'
VALIDATION_ORIGIN = pd.Timestamp('2024-04-01')
TIME_LIMIT_SECONDS = int(os.environ.get('AUTOGLUON_TIME_LIMIT', '600'))
RUN_ID = os.environ.get('AUTOGLUON_RUN_ID', pd.Timestamp.now().strftime('%Y%m%d_%H%M%S'))
MODEL_HYPERPARAMETERS = {'GBM': {}, 'CAT': {}}
COMPACT_FEATURES = [
    'TOWN', 'BLOCK', 'STREET', 'FLAT_TYPE', 'FLAT_MODEL', 'FLOOR_AREA_SQM',
    'approval_date_num', 'approval_month', 'coe_quota_premium_mean', 'YEAR_COMPLETED',
    'MAX_FLOOR', 'mrt_dist_nearest_km', 'mall_dist_nearest_km',
]


## Load TFT forecasts and build safe residual labels

The training residual baseline is the same-month **leave-one-out** segment mean, so a row never uses its own rent in its target-derived baseline. The validation and test baselines are TFT forecasts, never observed rents.

In [2]:
data_dir, datasets = load_datasets(project_root / 'data')
raw_train = categorical_clean(datasets['train'], columns=['TOWN', 'FLAT_TYPE'], context='hybrid inputs', log=False)
raw_test = categorical_clean(datasets['test'], columns=['TOWN', 'FLAT_TYPE'], context='hybrid inputs', log=False)
for frame in [raw_train, raw_test]:
    frame['period'] = pd.to_datetime(frame['RENT_APPROVAL_DATE'] + '-01')

validation_forecasts = pd.read_csv(experiment_dir / 'exp_9_dl_tft_validation_forecasts.csv', parse_dates=['period'])
final_forecasts = pd.read_csv(experiment_dir / 'exp_9_dl_tft_market_forecasts.csv', parse_dates=['period'])

def add_series_id(frame):
    result = frame.copy()
    result['series_id'] = result['TOWN'].astype(str) + ' | ' + result['FLAT_TYPE'].astype(str)
    return result

raw_train = add_series_id(raw_train)
raw_test = add_series_id(raw_test)

# Replace an observed segment mean with a leave-one-out mean for residual labels.
# Singleton segment-months fall back to their pre-origin segment mean.
def leave_one_out_market(frame):
    result = frame.copy()
    keys = ['series_id', 'period']
    stats = result.groupby(keys)[TARGET].agg(['sum', 'count']).rename(columns={'sum': '_sum', 'count': '_count'})
    result = result.join(stats, on=keys)
    result['market_loo'] = (result['_sum'] - result[TARGET]) / (result['_count'] - 1)
    prior = result.groupby('series_id')[TARGET].mean()
    result['market_loo'] = result['market_loo'].fillna(result['series_id'].map(prior))
    return result.drop(columns=['_sum', '_count'])

# Forecast tables must cover every record whose baseline they will supply.
validation_market = validation_forecasts[['series_id', 'period', 'predicted_rent']].rename(columns={'predicted_rent': 'market_forecast'})
final_market = final_forecasts[['series_id', 'period', 'PREDICTED_MARKET_RENT']].rename(columns={'PREDICTED_MARKET_RENT': 'market_forecast'})
fit_raw = raw_train[raw_train['period'] < VALIDATION_ORIGIN].copy()
valid_raw = raw_train[raw_train['period'] >= VALIDATION_ORIGIN].copy()
valid_raw = valid_raw.merge(validation_market, on=['series_id', 'period'], how='left', validate='many_to_one')
test_with_market = raw_test.merge(final_market, on=['series_id', 'period'], how='left', validate='many_to_one')
# TFT is deliberately trained only on segments with >=24 observed months.
# Preserve row-level coverage for short-history/unseen segments with a history-only fallback.
validation_fallback = fit_raw.groupby('series_id')[TARGET].mean()
valid_raw['market_forecast'] = valid_raw['market_forecast'].fillna(valid_raw['series_id'].map(validation_fallback)).fillna(fit_raw[TARGET].mean())
test_history = raw_train[raw_train['period'] < raw_test['period'].min()]
test_fallback = test_history.groupby('series_id')[TARGET].mean()
test_with_market['market_forecast'] = test_with_market['market_forecast'].fillna(test_with_market['series_id'].map(test_fallback)).fillna(test_history[TARGET].mean())
assert valid_raw['market_forecast'].notna().all() and test_with_market['market_forecast'].notna().all()
fit_raw = leave_one_out_market(fit_raw)
fit_raw['LOG_RESIDUAL'] = np.log(fit_raw[TARGET]) - np.log(fit_raw['market_loo'].clip(lower=1))
print(f'Validation rows: {len(valid_raw):,}; test rows with TFT baseline: {len(test_with_market):,}')


Validation rows: 35,158; test rows with TFT baseline: 50,000


In [3]:
## Fit direct and residual listing models on the same chronological split

def feature_frames(train_raw, other_raw):
    preprocessor = RentalPreprocessor(datasets)
    train_x = preprocessor.fit_transform(train_raw).sort_index()
    other_x = preprocessor.transform(other_raw).sort_index()
    return train_x[COMPACT_FEATURES], other_x[COMPACT_FEATURES]

fit_x, valid_x = feature_frames(fit_raw.drop(columns=['period', 'series_id', 'market_loo', 'LOG_RESIDUAL']), valid_raw.drop(columns=['period', 'series_id', 'market_forecast']))

def fit_model(label, x, y, name, tuning_x=None, tuning_y=None):
    predictor = TabularPredictor(
        label=label, problem_type='regression', eval_metric='root_mean_squared_error',
        path=str(experiment_dir / 'ag_exp_10' / RUN_ID / name),
    )
    kwargs = {} if tuning_x is None else {'tuning_data': tuning_x.assign(**{label: tuning_y.to_numpy()})}
    return predictor.fit(
        train_data=x.assign(**{label: y.to_numpy()}), hyperparameters=MODEL_HYPERPARAMETERS,
        presets='medium_quality', time_limit=TIME_LIMIT_SECONDS, verbosity=2, **kwargs,
    )

direct_model = fit_model(TARGET, fit_x, fit_raw[TARGET], 'validation_direct', valid_x, valid_raw[TARGET])
residual_model = fit_model('LOG_RESIDUAL', fit_x, fit_raw['LOG_RESIDUAL'], 'validation_residual')
direct_prediction = direct_model.predict(valid_x).astype(float).clip(lower=0).to_numpy()
hybrid_prediction = valid_raw['market_forecast'].to_numpy() * np.exp(residual_model.predict(valid_x).to_numpy())

# Learn only a simple convex blend from the chronological holdout.
y_valid = valid_raw[TARGET].to_numpy()
weights = np.linspace(0, 1, 21)
blend_scores = [(weight, mean_squared_error(y_valid, weight * hybrid_prediction + (1 - weight) * direct_prediction) ** 0.5) for weight in weights]
best_weight, _ = min(blend_scores, key=lambda row: row[1])
blended_prediction = best_weight * hybrid_prediction + (1 - best_weight) * direct_prediction

def metric_row(name, prediction):
    return {'model': name, 'RMSE': mean_squared_error(y_valid, prediction) ** 0.5,
            'MAE': mean_absolute_error(y_valid, prediction), 'R2': r2_score(y_valid, prediction)}
validation_metrics = pd.DataFrame([
    metric_row('direct_compact', direct_prediction),
    metric_row('tft_plus_residual', hybrid_prediction),
    metric_row(f'blend_tft_weight_{best_weight:.2f}', blended_prediction),
])
display(validation_metrics.round(3))
validation_metrics.to_csv(experiment_dir / 'exp_10_hybrid_validation_metrics.csv', index=False)


[categorical_clean] HDB lookup | TOWN: 26 -> 26 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | BLOCK: 2735 -> 2735 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | STREET: 576 -> 576 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | POSTAL_CODE: 9180 -> 9180 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | SUBZONE: 156 -> 156 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | PLANNING_AREA: 31 -> 31 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces


Verbosity: 2 (Standard Logging)


[categorical_clean] derived HDB feature | REGION: 5 -> 5 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces


=================== System Info ===================
AutoGluon Version:  1.6.2
Python Version:     3.12.14
Operating System:   Darwin
Platform Machine:   arm64
Platform Version:   Darwin Kernel Version 25.6.0: Fri Jul 31 19:16:20 PDT 2026; root:xnu-12377.161.14~5/RELEASE_ARM64_T8142
CPU Count:          10
Pytorch Version:    2.10.0
CUDA Version:       CUDA is not available
GPU Count:          WARNING: Exception was raised when calculating GPU count (AssertionError)
Memory Avail:       3.86 GB / 16.00 GB (24.1%)
Disk Space Avail:   185.44 GB / 460.38 GB (40.3%)
Presets specified: ['medium_quality']
Beginning AutoGluon training ... Time limit = 600s
AutoGluon will save models to "/Users/metul/Desktop/KDDM-project/exploration/metul/ag_exp_10/20260924_143232/validation_direct"
Train Data Rows:    114842
Train Data Columns: 13
Tuning Data Rows:    35158
Tuning Data Columns: 13
Label Column:       MONTHLY_RENT
Problem Type:       regression
Preprocessing data...
Using Feature Generators to pr

,model,RMSE,MAE,R2
0,direct_compact,471.209,349.000,0.489
1,tft_plus_residual,499.791,391.545,0.425
2,blend_tft_weight_0.00,471.209,349.000,0.489


## Refit on all labelled history before 2025-03 and predict every test listing

The March-2025 labelled rows are excluded because the test set also contains March-2025 records. The final prediction uses the validation-selected blend weight.

In [4]:
TEST_ORIGIN = raw_test['period'].min()
full_raw = raw_train[raw_train['period'] < TEST_ORIGIN].copy()
full_raw = leave_one_out_market(full_raw)
full_raw['LOG_RESIDUAL'] = np.log(full_raw[TARGET]) - np.log(full_raw['market_loo'].clip(lower=1))
full_x, test_x = feature_frames(
    full_raw.drop(columns=['period', 'series_id', 'market_loo', 'LOG_RESIDUAL']),
    test_with_market.drop(columns=['period', 'series_id', 'market_forecast']),
)
final_direct = fit_model(TARGET, full_x, full_raw[TARGET], 'final_direct')
final_residual = fit_model('LOG_RESIDUAL', full_x, full_raw['LOG_RESIDUAL'], 'final_residual')
test_direct = final_direct.predict(test_x).astype(float).clip(lower=0).to_numpy()
test_hybrid = test_with_market['market_forecast'].to_numpy() * np.exp(final_residual.predict(test_x).to_numpy())
test_prediction = (best_weight * test_hybrid + (1 - best_weight) * test_direct).clip(0, None)

diagnostic = pd.DataFrame({
    'Id': raw_test.index, 'TFT_MARKET_RENT': test_with_market['market_forecast'],
    'DIRECT_COMPACT_PREDICTION': test_direct, 'TFT_RESIDUAL_PREDICTION': test_hybrid,
    'PREDICTED_MONTHLY_RENT': test_prediction,
})
diagnostic.to_csv(experiment_dir / 'exp_10_hybrid_test_predictions.csv', index=False)
submission = diagnostic[['Id', 'PREDICTED_MONTHLY_RENT']].rename(columns={'PREDICTED_MONTHLY_RENT': 'Predicted'})
assert submission['Id'].is_unique and len(submission) == len(raw_test) and submission['Predicted'].notna().all()
submission.to_csv(experiment_dir / 'exp_10_hybrid_submission.csv', index=False)
print(f'Saved {len(submission):,} predictions; TFT blend weight = {best_weight:.2f}')
display(submission.head())


[categorical_clean] HDB lookup | TOWN: 26 -> 26 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | BLOCK: 2735 -> 2735 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | STREET: 576 -> 576 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | POSTAL_CODE: 9180 -> 9180 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | SUBZONE: 156 -> 156 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | PLANNING_AREA: 31 -> 31 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces


Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.6.2
Python Version:     3.12.14
Operating System:   Darwin
Platform Machine:   arm64
Platform Version:   Darwin Kernel Version 25.6.0: Fri Jul 31 19:16:20 PDT 2026; root:xnu-12377.161.14~5/RELEASE_ARM64_T8142
CPU Count:          10
Pytorch Version:    2.10.0
CUDA Version:       CUDA is not available
GPU Count:          WARNING: Exception was raised when calculating GPU count (AssertionError)
Memory Avail:       3.81 GB / 16.00 GB (23.8%)
Disk Space Avail:   185.36 GB / 460.38 GB (40.3%)
Presets specified: ['medium_quality']
Beginning AutoGluon training ... Time limit = 600s
AutoGluon will save models to "/Users/metul/Desktop/KDDM-project/exploration/metul/ag_exp_10/20260924_143232/final_direct"
Train Data Rows:    146968
Train Data Columns: 13
Label Column:       MONTHLY_RENT
Problem Type:       regression
Preprocessing data...
Using Feature Generators to preprocess the data ...
Fi

[categorical_clean] derived HDB feature | REGION: 5 -> 5 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces


	Stage 1 Generators:
		Fitting AsTypeFeatureGenerator...
	Stage 2 Generators:
		Fitting FillNaFeatureGenerator...
	Stage 3 Generators:
		Fitting IdentityFeatureGenerator...
		Fitting CategoryFeatureGenerator...
			Fitting CategoryMemoryMinimizeFeatureGenerator...
	Stage 4 Generators:
		Fitting DropUniqueFeatureGenerator...
	Stage 5 Generators:
		Fitting DropDuplicatesFeatureGenerator...
	Types of features in original data (raw dtype, special dtypes):
		('float', [])  : 4 | ['FLOOR_AREA_SQM', 'coe_quota_premium_mean', 'mrt_dist_nearest_km', 'mall_dist_nearest_km']
		('int', [])    : 4 | ['approval_date_num', 'approval_month', 'YEAR_COMPLETED', 'MAX_FLOOR']
		('object', []) : 5 | ['TOWN', 'BLOCK', 'STREET', 'FLAT_TYPE', 'FLAT_MODEL']
	Types of features in processed data (raw dtype, special dtypes):
		('category', []) : 5 | ['TOWN', 'BLOCK', 'STREET', 'FLAT_TYPE', 'FLAT_MODEL']
		('float', [])    : 4 | ['FLOOR_AREA_SQM', 'coe_quota_premium_mean', 'mrt_dist_nearest_km', 'mall_dist_nearest_

Saved 50,000 predictions; TFT blend weight = 0.00


,Id,Predicted
0,0,2699.188477
1,1,3585.414551
2,2,3211.229492
3,3,3171.850830
4,4,3241.038086
